# Análisis de Gaussian Splatting / Splatfacto

Este notebook analiza el output estructural de **Gaussian Splatting** a partir de un archivo `splat.ply` exportado desde Nerfstudio/Splatfacto.

## Objetivos

- caracterizar la representación explícita basada en **gaussianas 3D**
- medir cantidad de gaussianas, bounding box, distribución espacial y peso de archivo
- analizar **opacidad** y **escala** de las gaussianas
- extraer información útil del log de exportación si está disponible
- generar un **reporte PDF** y tablas `.csv` para usar en la tesis

## Estructura sugerida

```text
PROJECT_DIR/
  exports/
    paraguas-vicentelopez-splat/
      splat.ply
  logs/
    05_export_gaussian_splat_ply.log
  gaussian-splat-outputs/
```

> El notebook guarda todos los resultados en `gaussian-splat-outputs` dentro del root del proyecto.

In [8]:
# Si hace falta, instalá dependencias
!pip -q install plyfile reportlab scipy pandas matplotlib numpy

In [9]:
from pathlib import Path
import os
import re
import math
import zipfile
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from plyfile import PlyData
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

print('Librerías cargadas.')

Librerías cargadas.


In [12]:
# Explicitly unmount Google Drive if it's in a problematic state.
# The '|| true' ensures the cell doesn't fail if fusermount reports an error (e.g., if not mounted).
!fusermount -uz /content/drive || true

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

fusermount: failed to unmount /content/drive: Invalid argument


ValueError: Mountpoint must not already contain files

## 1. Configuración de rutas

Ajustá `PROJECT_DIR` a la carpeta raíz de tu proyecto en Drive o Colab.

In [4]:

# =========================
# CONFIGURACIÓN
# =========================
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Tesis: Tecnicas de CV para Reconstruccion Arquitectonica/notebooks')

OUTPUT_DIR = PROJECT_DIR / 'gaussian-splat-outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLAT_PLY_PATH = PROJECT_DIR / 'splat.ply'
EXPORT_LOG_PATH = PROJECT_DIR / '05_export_gaussian_splat_ply.log'

print('PROJECT_DIR =', PROJECT_DIR, '| exists:', PROJECT_DIR.exists())
print('OUTPUT_DIR  =', OUTPUT_DIR, '| exists:', OUTPUT_DIR.exists())
print('SPLAT_PLY_PATH =', SPLAT_PLY_PATH, '| exists:', SPLAT_PLY_PATH.exists())
print('EXPORT_LOG_PATH =', EXPORT_LOG_PATH, '| exists:', EXPORT_LOG_PATH.exists())

print('\\nArchivos encontrados en PROJECT_DIR:')
for p in sorted(PROJECT_DIR.iterdir()):
    print('-', p.name)

ValueError: Mountpoint must not already contain files

## 2. Búsqueda automática de archivos candidatos

Si la ruta exacta no coincide con tu estructura actual, esta celda busca archivos candidatos dentro del proyecto.

In [6]:
def list_candidates(project_dir, pattern, limit=30):
    files = sorted(project_dir.rglob(pattern))
    return files[:limit], len(files)

ply_candidates, ply_total = list_candidates(PROJECT_DIR, 'splat*.ply')
log_candidates, log_total = list_candidates(PROJECT_DIR, '*export*gaussian*.log')
if log_total == 0:
    log_candidates, log_total = list_candidates(PROJECT_DIR, '*05_export*.log')

print(f'Candidatos PLY encontrados: {ply_total}')
for p in ply_candidates:
    print(' -', p)

print(f'
Candidatos LOG encontrados: {log_total}')
for p in log_candidates:
    print(' -', p)

SyntaxError: unterminated f-string literal (detected at line 14) (880116756.py, line 14)

Si la ruta mostrada arriba es distinta, actualizá `SPLAT_PLY_PATH` y/o `EXPORT_LOG_PATH` manualmente y corré otra vez la celda de configuración.

In [ ]:
assert SPLAT_PLY_PATH.exists(), f'No existe el archivo PLY: {SPLAT_PLY_PATH}'
print('Archivo PLY encontrado correctamente.')
if EXPORT_LOG_PATH.exists():
    print('Log de exportación encontrado.')
else:
    print('Log de exportación no encontrado. El notebook seguirá igual.')

## 3. Cargar el archivo PLY y revisar su estructura

Este paso inventaría las propiedades presentes en la gaussiana exportada.

In [ ]:
ply = PlyData.read(str(SPLAT_PLY_PATH))
vertex = ply['vertex']
vertex_data = vertex.data
prop_names = list(vertex_data.dtype.names)

file_size_mb = SPLAT_PLY_PATH.stat().st_size / (1024**2)
gaussian_count = len(vertex_data)

inventory_df = pd.DataFrame({
    'property_name': prop_names,
    'dtype': [str(vertex_data.dtype.fields[name][0]) for name in prop_names]
})

summary_basic_df = pd.DataFrame([{
    'file_name': SPLAT_PLY_PATH.name,
    'file_path': str(SPLAT_PLY_PATH),
    'file_size_mb': file_size_mb,
    'gaussian_count': gaussian_count,
    'property_count': len(prop_names)
}])

print('PLY cargado.')
display(summary_basic_df)
display(inventory_df)

In [ ]:
summary_basic_df.to_csv(OUTPUT_DIR / 'gaussian_basic_summary.csv', index=False)
inventory_df.to_csv(OUTPUT_DIR / 'gaussian_property_inventory.csv', index=False)
print('CSV guardados en', OUTPUT_DIR)

## 4. Inferencia de columnas geométricas y atributos de gaussianas

Los exporters de Gaussian Splatting pueden usar nombres de propiedades ligeramente distintos. Esta celda intenta detectar automáticamente:

- posición (`x`, `y`, `z`)
- opacidad (`opacity`)
- escalas (`scale_0`, `scale_1`, `scale_2` u otras variantes)
- rotación / quaternion
- color / SH coefficients

In [ ]:
def first_existing(names, candidates):
    for c in candidates:
        if c in names:
            return c
    return None

pos_cols = [c for c in ['x', 'y', 'z'] if c in prop_names]
opacity_col = first_existing(prop_names, ['opacity', 'alpha', 'opacity_0'])

scale_cols = [c for c in prop_names if re.fullmatch(r'scale[_-]?\d+', c)]
if len(scale_cols) == 0:
    scale_cols = [c for c in prop_names if 'scale' in c.lower()]
scale_cols = sorted(scale_cols)

rot_cols = [c for c in prop_names if re.fullmatch(r'rot[_-]?\d+', c) or 'quat' in c.lower() or 'rotation' in c.lower()]
rot_cols = sorted(rot_cols)

color_cols = [c for c in prop_names if c in ['red', 'green', 'blue', 'r', 'g', 'b']]
sh_cols = [c for c in prop_names if c.startswith('f_dc_') or c.startswith('f_rest_')]

info_df = pd.DataFrame([
    {'attribute_group': 'position', 'columns': ', '.join(pos_cols) if pos_cols else ''},
    {'attribute_group': 'opacity', 'columns': opacity_col or ''},
    {'attribute_group': 'scale', 'columns': ', '.join(scale_cols) if scale_cols else ''},
    {'attribute_group': 'rotation', 'columns': ', '.join(rot_cols) if rot_cols else ''},
    {'attribute_group': 'rgb', 'columns': ', '.join(color_cols) if color_cols else ''},
    {'attribute_group': 'spherical_harmonics', 'columns': ', '.join(sh_cols[:10]) + (' ...' if len(sh_cols) > 10 else '')},
])

display(info_df)
info_df.to_csv(OUTPUT_DIR / 'gaussian_detected_attributes.csv', index=False)

## 5. Métricas espaciales globales

In [ ]:
assert len(pos_cols) == 3, 'No se pudieron detectar las columnas x, y, z.'

xyz = np.vstack([vertex_data[c] for c in pos_cols]).T.astype(np.float64)
mins = xyz.min(axis=0)
maxs = xyz.max(axis=0)
centroid = xyz.mean(axis=0)
extents = maxs - mins
bbox_volume = float(np.prod(extents))

spatial_df = pd.DataFrame([{
    'gaussian_count': len(xyz),
    'bbox_min_x': mins[0], 'bbox_min_y': mins[1], 'bbox_min_z': mins[2],
    'bbox_max_x': maxs[0], 'bbox_max_y': maxs[1], 'bbox_max_z': maxs[2],
    'extent_x': extents[0], 'extent_y': extents[1], 'extent_z': extents[2],
    'centroid_x': centroid[0], 'centroid_y': centroid[1], 'centroid_z': centroid[2],
    'bbox_volume': bbox_volume,
    'gaussians_per_bbox_unit': len(xyz) / bbox_volume if bbox_volume > 0 else np.nan,
}])

display(spatial_df.T)
spatial_df.to_csv(OUTPUT_DIR / 'gaussian_spatial_metrics.csv', index=False)

## 6. Interpretación de opacidad

En muchos exports de Gaussian Splatting, la propiedad `opacity` puede estar en **logit space**. Por eso el notebook calcula:

- valores crudos
- una estimación `sigmoid(opacity)` si los datos parecen no estar en `[0, 1]`

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

opacity_metrics_df = pd.DataFrame()
alpha = None

if opacity_col is not None:
    opacity_raw = np.asarray(vertex_data[opacity_col], dtype=np.float64)
    raw_min, raw_max = float(np.min(opacity_raw)), float(np.max(opacity_raw))

    if raw_min < 0 or raw_max > 1:
        alpha = sigmoid(opacity_raw)
        opacity_mode = 'logit_detected__sigmoid_applied'
    else:
        alpha = opacity_raw.copy()
        opacity_mode = 'already_in_0_1'

    opacity_metrics_df = pd.DataFrame([{
        'opacity_column': opacity_col,
        'opacity_mode': opacity_mode,
        'raw_min': raw_min,
        'raw_max': raw_max,
        'raw_mean': float(np.mean(opacity_raw)),
        'raw_std': float(np.std(opacity_raw)),
        'alpha_min': float(np.min(alpha)),
        'alpha_max': float(np.max(alpha)),
        'alpha_mean': float(np.mean(alpha)),
        'alpha_std': float(np.std(alpha)),
        'alpha_median': float(np.median(alpha)),
        'alpha_p05': float(np.percentile(alpha, 5)),
        'alpha_p25': float(np.percentile(alpha, 25)),
        'alpha_p75': float(np.percentile(alpha, 75)),
        'alpha_p95': float(np.percentile(alpha, 95)),
        'alpha_below_0_05': int(np.sum(alpha < 0.05)),
        'alpha_below_0_10': int(np.sum(alpha < 0.10)),
        'alpha_above_0_50': int(np.sum(alpha > 0.50)),
    }])

    display(opacity_metrics_df)
    opacity_metrics_df.to_csv(OUTPUT_DIR / 'gaussian_opacity_metrics.csv', index=False)
else:
    print('No se detectó columna de opacidad.')

In [ ]:
if opacity_col is not None and alpha is not None:
    plt.figure(figsize=(8, 5))
    plt.hist(alpha, bins=60)
    plt.xlabel('Opacidad estimada (alpha)')
    plt.ylabel('Cantidad de gaussianas')
    plt.title('Distribución de opacidad estimada')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'gaussian_opacity_histogram.png', dpi=150)
    plt.show()

## 7. Interpretación de escala

En muchos `.ply` de gaussianas, las columnas `scale_*` pueden estar en **espacio logarítmico**. El notebook calcula:

- valor crudo
- una estimación `exp(scale_raw)` para analizar tamaño relativo

In [ ]:
scale_metrics_df = pd.DataFrame()
scale_exp = None

if len(scale_cols) > 0:
    scale_raw = np.vstack([vertex_data[c] for c in scale_cols]).T.astype(np.float64)
    scale_exp = np.exp(scale_raw)

    rows = []
    for i, c in enumerate(scale_cols):
        rows.append({
            'scale_column': c,
            'raw_min': float(np.min(scale_raw[:, i])),
            'raw_max': float(np.max(scale_raw[:, i])),
            'raw_mean': float(np.mean(scale_raw[:, i])),
            'raw_std': float(np.std(scale_raw[:, i])),
            'exp_min': float(np.min(scale_exp[:, i])),
            'exp_max': float(np.max(scale_exp[:, i])),
            'exp_mean': float(np.mean(scale_exp[:, i])),
            'exp_std': float(np.std(scale_exp[:, i])),
            'exp_median': float(np.median(scale_exp[:, i])),
            'exp_p95': float(np.percentile(scale_exp[:, i], 95)),
        })
    scale_metrics_df = pd.DataFrame(rows)
    display(scale_metrics_df)
    scale_metrics_df.to_csv(OUTPUT_DIR / 'gaussian_scale_metrics.csv', index=False)
else:
    print('No se detectaron columnas de escala.')

In [ ]:
if len(scale_cols) > 0 and scale_exp is not None:
    plt.figure(figsize=(8, 5))
    for i, c in enumerate(scale_cols[:3]):
        plt.hist(scale_exp[:, i], bins=60, alpha=0.6, label=c)
    plt.xlabel('Escala estimada (exp(scale_raw))')
    plt.ylabel('Cantidad de gaussianas')
    plt.title('Distribución de escalas estimadas')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'gaussian_scale_histogram.png', dpi=150)
    plt.show()

## 8. Distribución espacial de gaussianas

Para no saturar la figura, se toma una muestra aleatoria reproducible.

In [ ]:
rng = np.random.default_rng(42)
sample_n = min(50000, len(xyz))
idx = rng.choice(len(xyz), size=sample_n, replace=False)
xyz_sample = xyz[idx]

plt.figure(figsize=(8, 6))
plt.scatter(xyz_sample[:, 0], xyz_sample[:, 1], s=0.2)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Distribución espacial XY (muestra)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gaussian_spatial_xy.png', dpi=150)
plt.show()

plt.figure(figsize=(8, 6))
plt.scatter(xyz_sample[:, 0], xyz_sample[:, 2], s=0.2)
plt.xlabel('X')
plt.ylabel('Z')
plt.title('Distribución espacial XZ (muestra)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gaussian_spatial_xz.png', dpi=150)
plt.show()

## 9. Distancia entre gaussianas vecinas más cercanas

Esta métrica no es una "densidad geométrica" exacta como en fotogrametría, pero ayuda a caracterizar cuán concentrada o dispersa es la representación.

In [ ]:
nn_metrics_df = pd.DataFrame()

sample_n_nn = min(20000, len(xyz))
idx_nn = rng.choice(len(xyz), size=sample_n_nn, replace=False)
xyz_nn = xyz[idx_nn]

if len(xyz_nn) >= 2:
    tree = cKDTree(xyz_nn)
    dists, _ = tree.query(xyz_nn, k=2)
    nn = dists[:, 1]

    nn_metrics_df = pd.DataFrame([{
        'sample_size': len(xyz_nn),
        'nn_min': float(np.min(nn)),
        'nn_max': float(np.max(nn)),
        'nn_mean': float(np.mean(nn)),
        'nn_std': float(np.std(nn)),
        'nn_median': float(np.median(nn)),
        'nn_p05': float(np.percentile(nn, 5)),
        'nn_p25': float(np.percentile(nn, 25)),
        'nn_p75': float(np.percentile(nn, 75)),
        'nn_p95': float(np.percentile(nn, 95)),
    }])
    display(nn_metrics_df)
    nn_metrics_df.to_csv(OUTPUT_DIR / 'gaussian_nearest_neighbor_metrics.csv', index=False)

    plt.figure(figsize=(8, 5))
    plt.hist(nn, bins=60)
    plt.xlabel('Distancia al vecino más cercano')
    plt.ylabel('Cantidad de gaussianas en la muestra')
    plt.title('Histograma de distancias entre gaussianas vecinas')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'gaussian_nn_histogram.png', dpi=150)
    plt.show()
else:
    print('No hay suficientes gaussianas para el cálculo NN.')

## 10. Parseo opcional del log de exportación

Esta celda intenta extraer datos útiles del log, por ejemplo:

- gaussianas exportadas / descartadas
- ruta de salida
- tiempos de proceso

In [ ]:
log_summary_rows = []

if EXPORT_LOG_PATH.exists():
    txt = EXPORT_LOG_PATH.read_text(encoding='utf-8', errors='ignore')

    patterns = {
        'total_gaussians': [r'(\d[\d\.,]*)\s+gaussians?\s+total', r'total[^
]*?(\d[\d\.,]*)'],
        'exported_gaussians': [r'(\d[\d\.,]*)\s+gaussians?\s+useful', r'(\d[\d\.,]*)\s+gaussians?\s+exported'],
        'discarded_gaussians': [r'discard(?:ed)?[^
]*?(\d[\d\.,]*)', r'low opacity[^
]*?(\d[\d\.,]*)'],
        'time_seconds': [r'real\s+(\d+)m([\d\.]+)s', r'Elapsed time[^
]*?([\d\.]+)\s+\[minutes\]'],
    }

    def parse_number(s):
        s = s.replace(',', '')
        try:
            return float(s)
        except:
            return None

    extracted = {}
    for key, pats in patterns.items():
        value = None
        for pat in pats:
            m = re.search(pat, txt, flags=re.IGNORECASE)
            if m:
                if key == 'time_seconds' and len(m.groups()) == 2:
                    value = int(m.group(1)) * 60 + float(m.group(2))
                elif key == 'time_seconds' and len(m.groups()) == 1:
                    value = float(m.group(1)) * 60
                else:
                    value = parse_number(m.group(1))
                break
        extracted[key] = value

    log_summary_rows.append({
        'log_file': str(EXPORT_LOG_PATH),
        **extracted
    })

    log_summary_df = pd.DataFrame(log_summary_rows)
    display(log_summary_df)
    log_summary_df.to_csv(OUTPUT_DIR / 'gaussian_export_log_summary.csv', index=False)
else:
    print('No se encontró log de exportación. Se omite este paso.')
    log_summary_df = pd.DataFrame()

## 11. Resumen interpretativo para tesis

In [ ]:
interpretation_rows = []

interpretation_rows.append({
    'dimension': 'representacion',
    'finding': 'El archivo analizado corresponde a una representación explícita basada en gaussianas 3D, no a una nube de puntos fotogramétrica clásica.',
})
interpretation_rows.append({
    'dimension': 'cantidad',
    'finding': f'Se analizaron {gaussian_count:,} gaussianas en el archivo {SPLAT_PLY_PATH.name}.',
})
interpretation_rows.append({
    'dimension': 'espacial',
    'finding': f'El bounding box estimado presenta extensiones aproximadas de X={extents[0]:.3f}, Y={extents[1]:.3f}, Z={extents[2]:.3f}.',
})
if opacity_col is not None and not opacity_metrics_df.empty:
    interpretation_rows.append({
        'dimension': 'opacidad',
        'finding': f'La opacidad estimada media es {opacity_metrics_df.iloc[0]["alpha_mean"]:.4f}, con {int(opacity_metrics_df.iloc[0]["alpha_below_0_10"]):,} gaussianas por debajo de 0.10.',
    })
if len(scale_cols) > 0 and not scale_metrics_df.empty:
    interpretation_rows.append({
        'dimension': 'escala',
        'finding': 'Las gaussianas presentan escalas variables, lo que permite adaptar el nivel de detalle y la continuidad visual de la escena.',
    })
if not nn_metrics_df.empty:
    interpretation_rows.append({
        'dimension': 'densidad_local',
        'finding': f'La distancia media al vecino más cercano en una muestra de {int(nn_metrics_df.iloc[0]["sample_size"]):,} gaussianas es {nn_metrics_df.iloc[0]["nn_mean"]:.6f}.',
    })
if 'log_summary_df' in globals() and not log_summary_df.empty:
    row = log_summary_df.iloc[0]
    if pd.notna(row.get('exported_gaussians')):
        interpretation_rows.append({
            'dimension': 'exportacion',
            'finding': f'El log de exportación reporta {int(row["exported_gaussians"]):,} gaussianas exportadas útiles.',
        })
    if pd.notna(row.get('discarded_gaussians')):
        interpretation_rows.append({
            'dimension': 'filtrado',
            'finding': f'Se descartaron aproximadamente {int(row["discarded_gaussians"]):,} gaussianas, presumiblemente por baja opacidad u otros filtros del exporter.',
        })

interpretation_df = pd.DataFrame(interpretation_rows)
display(interpretation_df)
interpretation_df.to_csv(OUTPUT_DIR / 'gaussian_interpretation_summary.csv', index=False)

## 12. Generar reporte PDF

In [ ]:
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as RLImage, PageBreak

PDF_PATH = OUTPUT_DIR / 'gaussian_splat_analysis_report.pdf'

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name='TitleCenter', parent=styles['Title'], alignment=TA_CENTER, fontSize=18, leading=22, spaceAfter=16))
styles.add(ParagraphStyle(name='SectionTitle', parent=styles['Heading1'], fontSize=14, leading=17, spaceBefore=10, spaceAfter=8))
styles.add(ParagraphStyle(name='BodySmall', parent=styles['BodyText'], fontSize=9, leading=12, spaceAfter=6))


def clean_text(value):
    if pd.isna(value):
        return ''
    return str(value).replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')


def make_table_pdf(df, font_size=7, max_rows=None):
    d = df.copy()
    if max_rows is not None:
        d = d.head(max_rows)
    for col in d.columns:
        if pd.api.types.is_float_dtype(d[col]):
            d[col] = d[col].round(4)
    data = [list(d.columns)] + d.astype(str).values.tolist()
    data = [[clean_text(cell) for cell in row] for row in data]
    table = Table(data, repeatRows=1)
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('GRID', (0, 0), (-1, -1), 0.25, colors.grey),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), font_size),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.whitesmoke]),
    ]))
    return table


def image_for_pdf(path, max_width=17*cm, max_height=11*cm):
    path = Path(path)
    if not path.exists():
        return None
    img = RLImage(str(path))
    iw, ih = img.imageWidth, img.imageHeight
    scale = min(max_width / iw, max_height / ih)
    img.drawWidth = iw * scale
    img.drawHeight = ih * scale
    return img

story = []
story.append(Paragraph('Análisis de Gaussian Splatting / Splatfacto', styles['TitleCenter']))
story.append(Paragraph('Caracterización estructural del archivo splat.ply', styles['BodySmall']))
story.append(Spacer(1, 0.4*cm))

story.append(Paragraph('1. Resumen básico', styles['SectionTitle']))
story.append(make_table_pdf(summary_basic_df, font_size=7))
story.append(Spacer(1, 0.3*cm))
story.append(make_table_pdf(spatial_df, font_size=7))

story.append(Paragraph('2. Atributos detectados', styles['SectionTitle']))
story.append(make_table_pdf(info_df, font_size=7))
story.append(Spacer(1, 0.3*cm))

if not opacity_metrics_df.empty:
    story.append(Paragraph('3. Métricas de opacidad', styles['SectionTitle']))
    story.append(make_table_pdf(opacity_metrics_df, font_size=6))
    img = image_for_pdf(OUTPUT_DIR / 'gaussian_opacity_histogram.png')
    if img is not None:
        story.append(Spacer(1, 0.2*cm))
        story.append(img)

story.append(PageBreak())

if not scale_metrics_df.empty:
    story.append(Paragraph('4. Métricas de escala', styles['SectionTitle']))
    story.append(make_table_pdf(scale_metrics_df, font_size=6))
    img = image_for_pdf(OUTPUT_DIR / 'gaussian_scale_histogram.png')
    if img is not None:
        story.append(Spacer(1, 0.2*cm))
        story.append(img)

story.append(Paragraph('5. Distribución espacial', styles['SectionTitle']))
for fn, title in [
    ('gaussian_spatial_xy.png', 'Distribución XY'),
    ('gaussian_spatial_xz.png', 'Distribución XZ'),
    ('gaussian_nn_histogram.png', 'Distancia al vecino más cercano'),
]:
    img = image_for_pdf(OUTPUT_DIR / fn)
    if img is not None:
        story.append(Paragraph(title, styles['BodySmall']))
        story.append(img)
        story.append(Spacer(1, 0.2*cm))

story.append(PageBreak())

story.append(Paragraph('6. Interpretación metodológica', styles['SectionTitle']))
story.append(make_table_pdf(interpretation_df, font_size=7, max_rows=20))

if 'log_summary_df' in globals() and not log_summary_df.empty:
    story.append(Spacer(1, 0.3*cm))
    story.append(Paragraph('7. Información extraída del log de exportación', styles['SectionTitle']))
    story.append(make_table_pdf(log_summary_df, font_size=7))

story.append(Spacer(1, 0.3*cm))
story.append(Paragraph(
    'Conclusión: el archivo splat.ply representa una estructura explícita de gaussianas 3D que privilegia continuidad visual, '
    'navegabilidad y fotorrealismo en tiempo real. Aunque su lógica no equivale a una nube de puntos fotogramétrica clásica ni a '
    'una malla BIM-ready, su análisis resulta fundamental para comprender la naturaleza de los outputs de Gaussian Splatting dentro '
    'de un pipeline comparativo con Fotogrametría y NeRF.',
    styles['BodySmall']
))

doc = SimpleDocTemplate(str(PDF_PATH), pagesize=A4, rightMargin=1.5*cm, leftMargin=1.5*cm, topMargin=1.5*cm, bottomMargin=1.5*cm)
doc.build(story)

print('PDF generado en:', PDF_PATH)

## 13. Empaquetar outputs en ZIP

In [ ]:
zip_path = OUTPUT_DIR / 'gaussian-splat-outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUTPUT_DIR.rglob('*')):
        if p.is_file() and p != zip_path:
            zf.write(p, arcname=p.relative_to(OUTPUT_DIR))

print('ZIP generado:', zip_path)
print('Archivos en OUTPUT_DIR:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(' -', p.name)